In [2]:
# Facebook Ads Dataset - Data Preprocessing and Feature Engineering
# Dataset: live_facebook_ads_dataset_v2.csv
# Author: Arman | UID: 12402573 | LPU

import ast
import numpy as np
import pandas as pd

# Load the dataset
df = pd.read_csv("live_facebook_ads_dataset_v2.csv")
print(df.shape)
print(df.head())

# Check missing values before cleaning
print(df.isnull().sum())


# ── 1. Handle Missing Data ────────────────────────────────────────────────────
# Fill missing ad text with "No Text"
df["ad_creative_bodies"] = df["ad_creative_bodies"].fillna("No Text")

# Fill missing bylines with "Unknown"
df["bylines"] = df["bylines"].fillna("Unknown")

# Verify no more missing values in these two columns
print(df[["ad_creative_bodies", "bylines"]].isnull().sum())


# ── 2. Parse spend, impressions, estimated_audience_size ─────────────────────
# These columns look like: {'lower_bound': '100', 'upper_bound': '199'}
# We need to convert them from string to dictionary using ast.literal_eval

# --- avg_spend ---
# First convert the string into a real dictionary
spend_list = []
for val in df["spend"]:
    d = ast.literal_eval(val)          # converts string -> dict
    lower = float(d["lower_bound"])
    # some rows only have lower_bound (e.g. top earners capped at 1000000+)
    if "upper_bound" in d:
        upper = float(d["upper_bound"])
        avg = (lower + upper) / 2      # take the midpoint
    else:
        avg = lower                    # just use lower_bound as-is
    spend_list.append(avg)

df["avg_spend"] = spend_list
print(df["avg_spend"].head())

# --- avg_impressions ---
impressions_list = []
for val in df["impressions"]:
    d = ast.literal_eval(val)
    lower = float(d["lower_bound"])
    if "upper_bound" in d:
        upper = float(d["upper_bound"])
        avg = (lower + upper) / 2
    else:
        avg = lower
    impressions_list.append(avg)

df["avg_impressions"] = impressions_list
print(df["avg_impressions"].head())

# --- min_audience_size ---
# estimated_audience_size only has lower_bound (e.g. {'lower_bound': '1000001'})
# So we just extract the lower_bound directly
audience_list = []
for val in df["estimated_audience_size"]:
    try:
        d = ast.literal_eval(val)
        lower = float(d["lower_bound"])
        audience_list.append(lower)
    except:
        audience_list.append(np.nan)   # handle the 1 missing value

df["min_audience_size"] = audience_list
print(df["min_audience_size"].head())


# ── 3. Extract Top Targeted Age and Gender from Demographics ──────────────────
# demographic_distribution looks like:
# [{'percentage': '0.53', 'age': '18-24', 'gender': 'male'}, ...]
# We want to find which group has the highest percentage

top_age_list = []
top_gender_list = []

for val in df["demographic_distribution"]:
    try:
        records = ast.literal_eval(val)   # convert string -> list of dicts

        # Find the group with highest percentage
        best = records[0]
        for r in records:
            if float(r["percentage"]) > float(best["percentage"]):
                best = r

        top_age_list.append(best["age"])
        top_gender_list.append(best["gender"])

    except:
        # If demographic data is missing or broken, put Unknown
        top_age_list.append("Unknown")
        top_gender_list.append("Unknown")

df["top_targeted_age"] = top_age_list
df["top_targeted_gender"] = top_gender_list

print(df["top_targeted_age"].value_counts())
print(df["top_targeted_gender"].value_counts())


# ── 4. One-Hot Encoding for Publisher Platforms ───────────────────────────────
# publisher_platforms looks like: ['facebook', 'instagram']
# We create two binary columns: on_facebook and on_instagram

facebook_list = []
instagram_list = []

for val in df["publisher_platforms"]:
    platforms = ast.literal_eval(val)   # convert string -> list

    if "facebook" in platforms:
        facebook_list.append(1)
    else:
        facebook_list.append(0)

    if "instagram" in platforms:
        instagram_list.append(1)
    else:
        instagram_list.append(0)

df["on_facebook"] = facebook_list
df["on_instagram"] = instagram_list

print(df["on_facebook"].value_counts())
print(df["on_instagram"].value_counts())


# ── 5. NLP Feature - Ad Text Length ──────────────────────────────────────────
# Count number of words in each ad
text_length_list = []

for text in df["ad_creative_bodies"]:
    if text == "No Text":
        text_length_list.append(0)     # ads with no text get 0
    else:
        words = text.split()           # split by spaces to get words
        text_length_list.append(len(words))

df["ad_text_length"] = text_length_list

print(df["ad_text_length"].describe())


# ── 6. Convert Datetime Column ────────────────────────────────────────────────
# Convert ad_delivery_start_time from string to proper datetime
df["ad_delivery_start_time"] = pd.to_datetime(df["ad_delivery_start_time"])

print(df["ad_delivery_start_time"].dtype)
print(df["ad_delivery_start_time"].head())


# ── Final Check ───────────────────────────────────────────────────────────────
print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum())


# Save the final cleaned dataset
df.to_csv("FINAL_facebook_ads_dataset.csv", index=False)
print("Done! File saved as FINAL_facebook_ads_dataset.csv")

(10000, 13)
                 id          page_id                   page_name  \
0  1391652419313737  428014347066247                  Hasib Khan   
1   941216855547919  428014347066247                  Hasib Khan   
2  1434665295122532  219673794876140                  DDNewsLive   
3   951319220731268  379784828740377  Doordarshan National (DD1)   
4   966152019427220  825425780803214            Shrinath Bhimale   

  ad_delivery_start_time                                         spend  \
0             2026-03-26     {'lower_bound': '0', 'upper_bound': '99'}   
1             2026-03-26     {'lower_bound': '0', 'upper_bound': '99'}   
2             2026-03-26  {'lower_bound': '200', 'upper_bound': '299'}   
3             2026-03-26  {'lower_bound': '400', 'upper_bound': '499'}   
4             2026-03-26  {'lower_bound': '100', 'upper_bound': '199'}   

                                        impressions currency  \
0        {'lower_bound': '0', 'upper_bound': '999'}      INR   
1     